# Tilted Multilayer Multislice

This notebook illustrates tilted-material voxelization for a multilayer sample defined by a recipe string, with no gold holography mask and light propagating along the simulation z direction. The incident field can be a plane wave or a Gaussian beam. The material planes are tilted by `theta_deg` relative to the simulation voxel planes. The extra z slices are vacuum, and interface voxels are fractional material/vacuum mixtures.

It also demonstrates the vector form of XMCD contrast. The circular magneto-optic term is longitudinal, so it follows `m . k`, where `m` is the local magnetization and `k` is the local light-momentum direction. With tilted illumination, the Jones tensor can be projected onto the fixed beam direction `normalize([tan(alpha_x), tan(alpha_y), 1])`. With `dielectric_tensor_local_k_projection=True`, the propagator instead estimates `k` before every material slice from the current Jones phase gradients, `Im(conj(E) grad E) / |E|^2`, so multislice diffraction can change the local contrast as the wavefront evolves through the stack.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if (ROOT / "src").exists():
    sys.path.insert(0, str(ROOT / "src"))
else:
    sys.path.insert(0, str(ROOT.parents[0] / "src"))

import importlib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.interpolate import RegularGridInterpolator

from scattering_calculator.database.database_loading import material_params
from scattering_calculator.sample_generator import structures as structures_module
structures_module = importlib.reload(structures_module)
Structure = structures_module.Structure
parse_recipe = structures_module.parse_recipe
from scattering_calculator.beam_propagator.Jones_propagator import wavefronts as JonesWavefronts
from scattering_calculator.experimental_conditions import light_beam
light_beam = importlib.reload(light_beam)

plt.rcParams.update({"figure.figsize": (12, 4), "image.cmap": "magma"})

## Build a Tilted Single-Layer Sample

`theta_deg = 0` recovers the untilted geometry. Increasing it expands the propagation stack because the tilted film needs vacuum before and after the material to fit inside planes perpendicular to the light direction.

In [ ]:
theta_deg = 0.0
tilt_axis = "x"
energy_ev = 781.14
real_space_pixel_size = 1.e-9
tilt_voxel_size = 2e-9

# User-defined simulation volume. The material stack is placed in this box.
simulation_size_x = 200e-9  # m
simulation_size_y = 200e-9  # m
simulation_size_z = 100e-9  # m

simulation_size_x = 200e-9  # m
simulation_size_y = 100e-9  # m
simulation_size_z = 900e-9  # m
# Grid counts are ceil(size / pixel_size); z slices are ceil(simulation_size_z / tilt_voxel_size).
# Large z volumes with sub-nm tilt_voxel_size can be slow and memory-heavy.
# Near theta=90 deg, keep simulation_size_z explicit and increase antialias_samples.
# Shift the slab centre relative to the centre of the simulation volume.
# Coordinates are lab-frame (y, x, z) in metres.
slab_center_offset = np.array([0.0, 0.0e-9, 0.0])

sample_recipe = "Co(890)"
# Adjacent terms like Pt(2)Co(1) become one effective-medium layer.
# Use "SiN(5)/[Pt(2)/Co(1)]x5" for separate Pt and Co propagation slices.

# Incident-field controls. Use "plane_wave" or "gaussian".
illumination_function = "gaussian"
illumination_center = np.array([0.0, 0.e-9])  # m, (y, x) relative to sample center
illumination_focus_distance = 0.0  # m, distance from waist to sample plane
illumination_fwhm = 50.0e-9  # m, Gaussian waist FWHM
illumination_alpha_beam = (0.0, np.deg2rad(0.0))  # rad, (alpha_y, alpha_x)
# If True, the Jones interaction recomputes local k from the phase gradient of
# the current electric field at every slice, so XMCD follows local m dot k.
# If False, contrast uses the nominal beam direction from illumination_alpha_beam.
use_local_k_projection = True


# Magnetic stripes are generated in coordinates attached to the tilted film.
# "y" means the stripes run along film_y, so m_z varies along film_x.
# In an x-z section, film interfaces are constant film_depth lines, while
# y-running stripe walls are constant film_x lines; those two families are not parallel.
magnetic_stripe_axis = "y"  # "x" or "y", direction parallel to the stripe length
magnetic_stripe_period = 8*real_space_pixel_size  # m
magnetic_stripe_smooth = False
# Shift the square-wave phase so domain walls fall between voxel centres.
# This avoids uneven domains when the period is an integer number of pixels.
magnetic_stripe_phase_offset = 0.5 * real_space_pixel_size  # m

# Triangular in-plane closure caps at the top and bottom film surfaces.
use_wedge_caps = True
wedge_x = 0*0.8e-9  # m, half-width of the triangular wedge at the film surface
wedge_z = 0*0.8e-9  # m, depth of the triangular wedge into the film
# Options: "+", "-", or "closure".  "+" and "-" force all caps along the
# positive/negative film-in-plane direction perpendicular to the stripes.
wedge_alignment = "closure"

# Overlay magnetic domain-wall and wedge-cap borders on the x-z exit-wave plots.
show_magnetic_borders_on_exit_slices = True





antialias_samples = 9  # increase near 90 degrees for smoother oblique interfaces
shape_yx = (
    int(np.ceil(simulation_size_y / real_space_pixel_size)),
    int(np.ceil(simulation_size_x / real_space_pixel_size)),
)



recipe = parse_recipe(sample_recipe)
recipe_materials = {material for layer in recipe.layers for material, _ in layer.components}
materials = material_params(materials=recipe_materials, x_ray_energy=energy_ev)
sample = Structure(
    name=sample_recipe,
    material_params=materials,
    sample_shape=[0, *shape_yx],
    real_space_pixel_size=real_space_pixel_size,
)
for layer in recipe.layers:
    if layer.is_composite:
        sample.add_effective_layer(layer.material, layer.components)
    else:
        sample.add_layer(layer.material, layer.thickness)

# No holography mask: every point in every material layer is present before tilt.
sample.mask = np.ones((len(sample.layer_names), *shape_yx), dtype=float)

# The regular per-layer magnetization remains defined for compatibility.
sample.magnetization = np.zeros((len(sample.layer_names), *shape_yx, 3), dtype=float)

sample.set_sample_tilt(
    theta=np.deg2rad(theta_deg),
    axis=tilt_axis,
    voxel_size=tilt_voxel_size,
    antialias_samples=antialias_samples,
    simulation_z_extent=simulation_size_z,
    center_offset=slab_center_offset,
)



film_y, film_x, film_depth = sample.tilted_material_coordinate_grids()
theta = np.deg2rad(theta_deg)
total_film_thickness = float(np.sum(sample.layer_thicknesses))

# Build the magnetic texture in an untilted coordinate system attached to the film,
# then interpolate it onto the tilted simulation voxels.  This avoids assigning
# stripes first and overwriting caps afterward on already-tilted interface voxels.
def centered_axis_covering(values, step):
    lo = np.floor(np.min(values) / step) * step
    hi = np.ceil(np.max(values) / step) * step
    n = int(np.ceil((hi - lo) / step)) + 1
    return lo + np.arange(n, dtype=float) * step

film_x_axis = centered_axis_covering(film_x, real_space_pixel_size)
film_y_axis = centered_axis_covering(film_y, real_space_pixel_size)
film_depth_axis = np.arange(
    0.0,
    total_film_thickness + 0.5 * tilt_voxel_size,
    tilt_voxel_size,
)
if film_depth_axis[-1] < total_film_thickness:
    film_depth_axis = np.append(film_depth_axis, total_film_thickness)
else:
    film_depth_axis[-1] = total_film_thickness

depth_grid = film_depth_axis[:, None, None]
y_grid = film_y_axis[None, :, None]
x_grid = film_x_axis[None, None, :]

if magnetic_stripe_axis == "y":
    stripe_coordinate_untilted = x_grid
    cap_component = 0  # film x, perpendicular to y-running stripes
    if tilt_axis == "x":
        cap_direction = np.array([np.cos(theta), 0.0, -np.sin(theta)])
    else:
        cap_direction = np.array([1.0, 0.0, 0.0])
elif magnetic_stripe_axis == "x":
    stripe_coordinate_untilted = y_grid
    cap_component = 1  # film y, perpendicular to x-running stripes
    if tilt_axis == "y":
        cap_direction = np.array([0.0, np.cos(theta), -np.sin(theta)])
    else:
        cap_direction = np.array([0.0, 1.0, 0.0])
else:
    raise ValueError("magnetic_stripe_axis must be 'x' or 'y'.")

stripe_coordinate_shifted = stripe_coordinate_untilted + magnetic_stripe_phase_offset
stripe_phase_untilted = 2.0 * np.pi * stripe_coordinate_shifted / magnetic_stripe_period
if magnetic_stripe_smooth:
    m_normal_untilted = np.sin(stripe_phase_untilted)
else:
    domain_width = 0.5 * magnetic_stripe_period
    domain_index = np.floor(stripe_coordinate_shifted / domain_width).astype(int)
    m_normal_untilted = np.where(domain_index % 2 == 0, 1.0, -1.0)
m_normal_untilted = np.broadcast_to(
    m_normal_untilted,
    (len(film_depth_axis), len(film_y_axis), len(film_x_axis)),
)

# Film-frame components are (film_x, film_y, film_normal).  The normal component
# carries the stripe domains; optional caps replace it with in-plane closure.
magnetization_film = np.zeros((*m_normal_untilted.shape, 3), dtype=float)
magnetization_film[..., 2] = m_normal_untilted

if use_wedge_caps and wedge_x > 0 and wedge_z > 0:
    wall_period = 0.5 * magnetic_stripe_period
    distance_to_wall = (
        stripe_coordinate_shifted + 0.5 * wall_period
    ) % wall_period - 0.5 * wall_period
    bottom_depth = depth_grid
    top_depth = total_film_thickness - depth_grid
    bottom_half_width = wedge_x * np.clip(1.0 - bottom_depth / wedge_z, 0.0, 1.0)
    top_half_width = wedge_x * np.clip(1.0 - top_depth / wedge_z, 0.0, 1.0)
    bottom_cap = (
        (bottom_depth >= 0.0)
        & (bottom_depth <= wedge_z)
        & (np.abs(distance_to_wall) <= bottom_half_width)
    )
    top_cap = (
        (top_depth >= 0.0)
        & (top_depth <= wedge_z)
        & (np.abs(distance_to_wall) <= top_half_width)
    )

    if wedge_alignment == "+":
        bottom_sign = np.ones_like(distance_to_wall)
        top_sign = np.ones_like(distance_to_wall)
    elif wedge_alignment == "-":
        bottom_sign = -np.ones_like(distance_to_wall)
        top_sign = -np.ones_like(distance_to_wall)
    elif wedge_alignment == "closure":
        wall_sign = np.sign(np.cos(stripe_phase_untilted))
        wall_sign[wall_sign == 0] = 1.0
        bottom_sign = +wall_sign
        top_sign = -wall_sign
    else:
        raise ValueError("wedge_alignment must be '+', '-', or 'closure'.")

    bottom_cap_full = np.broadcast_to(bottom_cap, m_normal_untilted.shape)
    top_cap_full = np.broadcast_to(top_cap, m_normal_untilted.shape)
    bottom_sign_full = np.broadcast_to(bottom_sign, m_normal_untilted.shape)
    top_sign_full = np.broadcast_to(top_sign, m_normal_untilted.shape)
    cap_region = bottom_cap_full | top_cap_full
    cap_sign = np.where(top_cap_full, top_sign_full, np.where(bottom_cap_full, bottom_sign_full, 0.0))
    magnetization_film[cap_region, 2] = 0.0
    magnetization_film[cap_region, cap_component] = cap_sign[cap_region]

# Interpolate the untilted film texture onto the tilted simulation voxel centers.
points = np.column_stack((film_depth.ravel(), film_y.ravel(), film_x.ravel()))
magnetization_sampled_film = np.empty((*film_depth.shape, 3), dtype=float)
for component_idx in range(3):
    interpolator = RegularGridInterpolator(
        (film_depth_axis, film_y_axis, film_x_axis),
        magnetization_film[..., component_idx],
        method="linear",
        bounds_error=False,
        fill_value=0.0,
    )
    magnetization_sampled_film[..., component_idx] = interpolator(points).reshape(film_depth.shape)

inside_film = (film_depth >= 0.0) & (film_depth <= total_film_thickness)
magnetization_sampled_film[~inside_film] = 0.0

# Rotate film-frame vectors into lab/simulation coordinates.
sample.tilted_magnetization = np.zeros((*film_depth.shape, 3), dtype=float)
if tilt_axis == "x":
    sample.tilted_magnetization[..., 0] = (
        magnetization_sampled_film[..., 0] * np.cos(theta)
        + magnetization_sampled_film[..., 2] * np.sin(theta)
    )
    sample.tilted_magnetization[..., 1] = magnetization_sampled_film[..., 1]
    sample.tilted_magnetization[..., 2] = (
        -magnetization_sampled_film[..., 0] * np.sin(theta)
        + magnetization_sampled_film[..., 2] * np.cos(theta)
    )
elif tilt_axis == "y":
    sample.tilted_magnetization[..., 0] = magnetization_sampled_film[..., 0]
    sample.tilted_magnetization[..., 1] = (
        magnetization_sampled_film[..., 1] * np.cos(theta)
        + magnetization_sampled_film[..., 2] * np.sin(theta)
    )
    sample.tilted_magnetization[..., 2] = (
        -magnetization_sampled_film[..., 1] * np.sin(theta)
        + magnetization_sampled_film[..., 2] * np.cos(theta)
    )
else:
    raise ValueError("tilt_axis must be 'x' or 'y'.")

polarization = "CR"
# Jones + beam-direction projection: magnetic contrast depends on m dot k and
# linear dichroism is projected into the polarization basis transverse to k.
# The direction is in lab (x, y, z) coordinates and follows the same alpha_beam
# convention used by the Gaussian illumination.
contrast_beam_direction = light_beam.beam_direction_from_alpha(illumination_alpha_beam)
sample.calculate_final_dielectric_tensor(
    compact=False,
    beam_direction=contrast_beam_direction,
    local_k_projection=use_local_k_projection,
)

eps_stack = sample.final_dielectric_tensor
# Display-only effective refractive index, averaged over the two transverse
# Jones axes.  Propagation below uses eps_stack directly; if local-k projection
# is active, eps_stack is dynamic and this materialized tensor is only a nominal
# beam-direction view for plotting.
eps_stack_display = eps_stack.materialize() if hasattr(eps_stack, "project_layer") else eps_stack
n_stack = np.sqrt(0.5 * (eps_stack_display[..., 0, 0] + eps_stack_display[..., 1, 1]))
thicknesses = np.asarray(sample.propagation_layer_thicknesses)
z_nm = (np.arange(eps_stack.shape[0]) - eps_stack.shape[0] / 2 + 0.5) * tilt_voxel_size * 1e9
x_nm = (np.arange(shape_yx[1]) - shape_yx[1] / 2 + 0.5) * real_space_pixel_size * 1e9
y_nm = (np.arange(shape_yx[0]) - shape_yx[0] / 2 + 0.5) * real_space_pixel_size * 1e9
y0 = shape_yx[0] // 2

layer_fractions, _ = sample._tilted_layer_fractions()
material_fraction = np.zeros(layer_fractions.shape[:3], dtype=float)
for layer_idx in range(len(sample.layer_names)):
    material_fraction += layer_fractions[..., layer_idx] * sample.mask[layer_idx]
material_fraction = np.clip(material_fraction, 0.0, 1.0)
magnetic_domain_overlay = np.ma.masked_where(
    material_fraction <= 0.05,
    magnetization_sampled_film[..., 2],
)
wedge_cap_overlay = np.ma.masked_where(
    material_fraction <= 0.05,
    np.abs(magnetization_sampled_film[..., cap_component]),
)

print(f"recipe = {sample_recipe}")
print(f"layers = {list(zip(sample.layer_names, np.asarray(sample.layer_thicknesses) * 1e9))}  # thickness in nm")
print(f"theta = {theta_deg:.1f} deg")
print(f"slab centre offset (y, x, z) = {np.asarray(slab_center_offset) * 1e9} nm")
print(f"contrast beam direction (x, y, z) = {contrast_beam_direction}")
print(f"local-k projection = {use_local_k_projection}")
print(f"grid shape (z, y, x) = {eps_stack.shape[:3]}")
print(f"simulation size (x, y, z) = ({shape_yx[1] * real_space_pixel_size * 1e9:.1f}, {shape_yx[0] * real_space_pixel_size * 1e9:.1f}, {thicknesses.sum() * 1e9:.1f}) nm")
print(f"fractional interface voxels = {np.count_nonzero((material_fraction > 0) & (material_fraction < 1))}")

In [ ]:
%matplotlib widget

z0 = n_stack.shape[0] // 2
x0 = shape_yx[1] // 2
film_depth_mid = 0.5 * np.sum(sample.layer_thicknesses)
depth_idx = np.argmin(np.abs(film_depth[:, y0, x0] - film_depth_mid))
material_mask = material_fraction > 0.05


def mumax_magnetization_rgb(
    vectors,
    mask=None,
    oop_component=2,
    ip_components=(0, 1),
    background=(0.82, 0.82, 0.82),
):
    """Mumax-style color map: hue is in-plane angle, brightness is OOP sign."""
    vectors = np.asarray(vectors, dtype=float)
    mx = vectors[..., ip_components[0]]
    my = vectors[..., ip_components[1]]
    mz = np.clip(vectors[..., oop_component], -1.0, 1.0)

    hue = (np.arctan2(my, mx) / (2.0 * np.pi) + 1.0) % 1.0
    saturation = np.clip(np.hypot(mx, my), 0.0, 1.0)
    value = np.clip(0.5 + 0.5 * mz, 0.0, 1.0)
    rgb = mcolors.hsv_to_rgb(np.stack((hue, saturation, value), axis=-1))

    if mask is not None:
        valid = np.asarray(mask, dtype=bool)
        rgb = np.where(valid[..., None], rgb, np.asarray(background, dtype=float))
    return rgb


def add_mumax_note(ax, frame_label):
    ax.text(
        0.02,
        0.02,
        f"{frame_label}\nBW: OOP -/+  hue: in-plane angle",
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=8,
        color="k",
        bbox={"boxstyle": "round,pad=0.25", "fc": "white", "ec": "none", "alpha": 0.78},
    )


# Semantic direction arrows: one at each stripe centre and one at each wedge centre.
def add_vector_arrows(ax, x_pos, y_pos, u, v, color="k", label=None, arrow_length_nm=None):
    x_pos = np.asarray(x_pos, dtype=float)
    y_pos = np.asarray(y_pos, dtype=float)
    u = np.asarray(u, dtype=float)
    v = np.asarray(v, dtype=float)
    norm = np.hypot(u, v)
    good = norm > 1e-8
    if not np.any(good):
        return
    if arrow_length_nm is None:
        arrow_length_nm = 0.28 * magnetic_stripe_period * 1e9
    uq = np.full_like(u, np.nan, dtype=float)
    vq = np.full_like(v, np.nan, dtype=float)
    uq[good] = arrow_length_nm * u[good] / norm[good]
    vq[good] = arrow_length_nm * v[good] / norm[good]
    ax.quiver(
        x_pos[good],
        y_pos[good],
        uq[good],
        vq[good],
        angles="xy",
        scale_units="xy",
        scale=1,
        color=color,
        pivot="middle",
        width=0.004,
        headwidth=4.0,
        headlength=5.0,
        headaxislength=4.5,
        label=label,
    )


def positions_in_range(start, stop, spacing, phase=0.0):
    first = np.floor((start - phase) / spacing) * spacing + phase
    values = first + np.arange(int(np.ceil((stop - first) / spacing)) + 2) * spacing
    return values[(values >= start) & (values <= stop)]


def sample_film_vectors(depth_values, film_y_values, film_x_values):
    query = np.column_stack((depth_values, film_y_values, film_x_values))
    sampled = np.empty((len(query), 3), dtype=float)
    for component_idx in range(3):
        interp = RegularGridInterpolator(
            (film_depth_axis, film_y_axis, film_x_axis),
            magnetization_film[..., component_idx],
            method="linear",
            bounds_error=False,
            fill_value=0.0,
        )
        sampled[:, component_idx] = interp(query)
    return sampled


def film_vectors_to_lab_xz(vectors_film):
    if tilt_axis == "x":
        u_lab = vectors_film[:, 0] * np.cos(theta) + vectors_film[:, 2] * np.sin(theta)
        v_lab = -vectors_film[:, 0] * np.sin(theta) + vectors_film[:, 2] * np.cos(theta)
    else:
        u_lab = vectors_film[:, 0]
        v_lab = -vectors_film[:, 1] * np.sin(theta) + vectors_film[:, 2] * np.cos(theta)
    return u_lab, v_lab


def film_points_to_lab_xz(film_coord_values, depth_values):
    depth_from_center = depth_values - 0.5 * total_film_thickness
    if magnetic_stripe_axis == "y":
        film_x_values = film_coord_values
    else:
        film_x_values = np.zeros_like(film_coord_values)
    if tilt_axis == "x":
        x_values = film_x_values * np.cos(theta) + depth_from_center * np.sin(theta)
        z_values = -film_x_values * np.sin(theta) + depth_from_center * np.cos(theta)
    else:
        y_lab = y_nm[y0] * 1e-9
        x_values = film_x_values
        if abs(np.cos(theta)) < 1e-9:
            z_values = np.full_like(x_values, np.nan)
        else:
            z_values = (depth_values - 0.5 * total_film_thickness - y_lab * np.sin(theta)) / np.cos(theta)
    return x_values * 1e9, z_values * 1e9


def semantic_arrow_points():
    if magnetic_stripe_axis == "y":
        coord_min = max(np.min(film_x[:, y0, :]), np.min(film_x_axis))
        coord_max = min(np.max(film_x[:, y0, :]), np.max(film_x_axis))
        wall_period = 0.5 * magnetic_stripe_period
        stripe_centers = positions_in_range(
            coord_min,
            coord_max,
            wall_period,
            -magnetic_stripe_phase_offset + 0.5 * wall_period,
        )
        wall_centers = positions_in_range(
            coord_min, coord_max, wall_period, -magnetic_stripe_phase_offset
        )
        stripe_y = np.zeros_like(stripe_centers)
        wall_y = np.zeros_like(wall_centers)
        stripe_x = stripe_centers
        wall_x = wall_centers
    else:
        coord_min = max(np.min(film_y[:, :, x0]), np.min(film_y_axis))
        coord_max = min(np.max(film_y[:, :, x0]), np.max(film_y_axis))
        wall_period = 0.5 * magnetic_stripe_period
        stripe_centers = positions_in_range(
            coord_min,
            coord_max,
            wall_period,
            -magnetic_stripe_phase_offset + 0.5 * wall_period,
        )
        wall_centers = positions_in_range(
            coord_min, coord_max, wall_period, -magnetic_stripe_phase_offset
        )
        stripe_y = stripe_centers
        wall_y = wall_centers
        stripe_x = np.zeros_like(stripe_centers)
        wall_x = np.zeros_like(wall_centers)

    stripe_depth = np.full_like(stripe_centers, film_depth_mid)
    stripe_vectors = sample_film_vectors(stripe_depth, stripe_y, stripe_x)

    if use_wedge_caps and wedge_x > 0 and wedge_z > 0 and len(wall_centers) > 0:
        cap_depth = min(max(wedge_z / 3.0, 0.0), total_film_thickness / 2.0)
        bottom_depth = np.full_like(wall_centers, cap_depth)
        top_depth = np.full_like(wall_centers, total_film_thickness - cap_depth)
        wedge_depth = np.concatenate((bottom_depth, top_depth))
        wedge_x_pos = np.concatenate((wall_x, wall_x))
        wedge_y_pos = np.concatenate((wall_y, wall_y))
        wedge_coord = np.concatenate((wall_centers, wall_centers))
        wedge_vectors = sample_film_vectors(wedge_depth, wedge_y_pos, wedge_x_pos)
    else:
        wedge_depth = np.array([], dtype=float)
        wedge_coord = np.array([], dtype=float)
        wedge_vectors = np.empty((0, 3), dtype=float)

    return stripe_centers, stripe_depth, stripe_vectors, wedge_coord, wedge_depth, wedge_vectors


stripe_coord, stripe_depth, stripe_vectors_film, wedge_coord, wedge_depth, wedge_vectors_film = semantic_arrow_points()
stripe_xz_x, stripe_xz_z = film_points_to_lab_xz(stripe_coord, stripe_depth)
wedge_xz_x, wedge_xz_z = film_points_to_lab_xz(wedge_coord, wedge_depth)
stripe_xz_u, stripe_xz_v = film_vectors_to_lab_xz(stripe_vectors_film)
wedge_xz_u, wedge_xz_v = film_vectors_to_lab_xz(wedge_vectors_film)
valid_stripe_xz = np.isfinite(stripe_xz_x) & np.isfinite(stripe_xz_z)
valid_wedge_xz = np.isfinite(wedge_xz_x) & np.isfinite(wedge_xz_z)

lab_z_rgb = mumax_magnetization_rgb(
    sample.tilted_magnetization[z0], mask=material_mask[z0]
)
lab_xz_rgb = mumax_magnetization_rgb(
    sample.tilted_magnetization[:, y0, :, :], mask=material_mask[:, y0, :]
)
film_xz_rgb = mumax_magnetization_rgb(
    magnetization_sampled_film[:, y0, :, :], mask=material_mask[:, y0, :]
)
film_mid_rgb = mumax_magnetization_rgb(
    magnetization_sampled_film[depth_idx], mask=material_mask[depth_idx]
)

fig, ax = plt.subplots(2, 2, figsize=(11, 8), constrained_layout=True)
ax[0, 0].imshow(
    lab_z_rgb,
    origin="lower",
    extent=[x_nm[0], x_nm[-1], y_nm[0], y_nm[-1]],
)
ax[0, 0].contour(x_nm, y_nm, material_fraction[z0], levels=[0.5], colors="w", linewidths=0.8)
ax[0, 0].set_title("lab-frame m on central z plane")
ax[0, 0].set_xlabel("lab x (nm)")
ax[0, 0].set_ylabel("lab y (nm)")
add_mumax_note(ax[0, 0], "lab frame")

ax[0, 1].imshow(
    lab_xz_rgb,
    origin="lower",
    aspect=1,
    extent=[x_nm[0], x_nm[-1], z_nm[0], z_nm[-1]],
)
ax[0, 1].contour(x_nm, z_nm, material_fraction[:, y0, :], levels=[0.5], colors="w", linewidths=0.8)
add_vector_arrows(
    ax[0, 1],
    stripe_xz_x[valid_stripe_xz],
    stripe_xz_z[valid_stripe_xz],
    stripe_xz_u[valid_stripe_xz],
    stripe_xz_v[valid_stripe_xz],
    color="k",
    label="stripe centres",
)
add_vector_arrows(
    ax[0, 1],
    wedge_xz_x[valid_wedge_xz],
    wedge_xz_z[valid_wedge_xz],
    wedge_xz_u[valid_wedge_xz],
    wedge_xz_v[valid_wedge_xz],
    color="lime",
    label="wedge centres",
)
ax[0, 1].set_title("lab-frame m, x-z section")
ax[0, 1].set_xlabel("lab x (nm)")
ax[0, 1].set_ylabel("propagation z (nm)")
add_mumax_note(ax[0, 1], "lab frame")

ax[1, 0].imshow(
    film_xz_rgb,
    origin="lower",
    aspect=1,
    extent=[x_nm[0], x_nm[-1], z_nm[0], z_nm[-1]],
)
ax[1, 0].contour(x_nm, z_nm, material_fraction[:, y0, :], levels=[0.5], colors="w", linewidths=0.8)
add_vector_arrows(
    ax[1, 0],
    stripe_xz_x[valid_stripe_xz],
    stripe_xz_z[valid_stripe_xz],
    stripe_xz_u[valid_stripe_xz],
    stripe_xz_v[valid_stripe_xz],
    color="k",
)
add_vector_arrows(
    ax[1, 0],
    wedge_xz_x[valid_wedge_xz],
    wedge_xz_z[valid_wedge_xz],
    wedge_xz_u[valid_wedge_xz],
    wedge_xz_v[valid_wedge_xz],
    color="lime",
)
ax[1, 0].set_title("film-frame m, x-z section")
ax[1, 0].set_xlabel("lab x (nm)")
ax[1, 0].set_ylabel("propagation z (nm)")
add_mumax_note(ax[1, 0], "film frame")

# This is the clean check: in film coordinates the stripe texture should be untilted.
film_x_depth_nm = film_x[depth_idx] * 1e9
film_y_depth_nm = film_y[depth_idx] * 1e9


def nondegenerate_extent(values):
    lo = float(np.min(values))
    hi = float(np.max(values))
    if np.isclose(lo, hi):
        lo -= 0.5
        hi += 0.5
    return lo, hi


film_x_min, film_x_max = nondegenerate_extent(film_x_depth_nm)
film_y_min, film_y_max = nondegenerate_extent(film_y_depth_nm)
ax[1, 1].imshow(
    film_mid_rgb,
    origin="lower",
    extent=[film_x_min, film_x_max, film_y_min, film_y_max],
)
if np.ptp(film_x_depth_nm[0]) > 0 and np.ptp(film_y_depth_nm[:, 0]) > 0:
    if magnetic_stripe_axis == "y":
        stripe_plot_x = stripe_coord * 1e9
        stripe_plot_y = np.zeros_like(stripe_coord)
        wedge_plot_x = wedge_coord * 1e9
        wedge_plot_y = np.zeros_like(wedge_coord)
    else:
        stripe_plot_x = np.zeros_like(stripe_coord)
        stripe_plot_y = stripe_coord * 1e9
        wedge_plot_x = np.zeros_like(wedge_coord)
        wedge_plot_y = wedge_coord * 1e9
    add_vector_arrows(
        ax[1, 1],
        stripe_plot_x,
        stripe_plot_y,
        stripe_vectors_film[:, 0],
        stripe_vectors_film[:, 1],
        color="k",
    )
    add_vector_arrows(
        ax[1, 1],
        wedge_plot_x,
        wedge_plot_y,
        wedge_vectors_film[:, 0],
        wedge_vectors_film[:, 1],
        color="lime",
    )
else:
    ax[1, 1].text(
        0.5,
        0.05,
        "constant-z slice is degenerate in film coordinates",
        transform=ax[1, 1].transAxes,
        ha="center",
        va="bottom",
        fontsize=8,
        color="k",
    )
ax[1, 1].set_title("film-frame m on mid-depth plane")
ax[1, 1].set_xlabel("film x (nm)")
ax[1, 1].set_ylabel("film y (nm)")
add_mumax_note(ax[1, 1], "film frame")
plt.show()


In [ ]:
# Choose what to display from the complex refractive index stack.
# Good options: 1 - np.real(n_stack), np.imag(n_stack), or np.real(n_stack).
refractive_index_volume = 1.0 - np.real(n_stack)
refractive_index_label = "1 - Re(n)"
x0 = shape_yx[1] // 2

# Raw index sections show the tilted material/vacuum geometry.
xz_section = refractive_index_volume[:, y0, :]
yz_section = refractive_index_volume[:, :, x0]

# Contrast sections suppress the z-dependent material/vacuum background,
# making magnetic stripe modulation easier to see.
xz_stripe_contrast = xz_section - np.mean(xz_section, axis=1, keepdims=True)
yz_stripe_contrast = yz_section - np.mean(yz_section, axis=1, keepdims=True)
contrast_limit = max(np.max(np.abs(xz_stripe_contrast)), np.max(np.abs(yz_stripe_contrast)))

fig, ax = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
im00 = ax[0, 0].imshow(
    xz_section,
    origin="lower",
    aspect=1,
    extent=[x_nm[0], x_nm[-1], z_nm[0], z_nm[-1]],
    cmap="viridis",
)
ax[0, 0].set_title("x-z refractive index at central y")
ax[0, 0].set_xlabel("x (nm)")
ax[0, 0].set_ylabel("propagation z (nm)")
fig.colorbar(im00, ax=ax[0, 0], label=refractive_index_label)

im01 = ax[0, 1].imshow(
    yz_section,
    origin="lower",
    aspect=1,
    extent=[y_nm[0], y_nm[-1], z_nm[0], z_nm[-1]],
    cmap="viridis",
)
ax[0, 1].set_title("y-z refractive index at central x")
ax[0, 1].set_xlabel("y (nm)")
ax[0, 1].set_ylabel("propagation z (nm)")
fig.colorbar(im01, ax=ax[0, 1], label=refractive_index_label)

im10 = ax[1, 0].imshow(
    xz_stripe_contrast,
    origin="lower",
    aspect=1,
    extent=[x_nm[0], x_nm[-1], z_nm[0], z_nm[-1]],
    cmap="coolwarm",
    vmin=-contrast_limit,
    vmax=contrast_limit,
)
ax[1, 0].set_title("x-z stripe contrast")
ax[1, 0].set_xlabel("x (nm)")
ax[1, 0].set_ylabel("propagation z (nm)")
fig.colorbar(im10, ax=ax[1, 0], label=f"{refractive_index_label} minus z-slice mean")

im11 = ax[1, 1].imshow(
    yz_stripe_contrast,
    origin="lower",
    aspect=1,
    extent=[y_nm[0], y_nm[-1], z_nm[0], z_nm[-1]],
    cmap="coolwarm",
    vmin=-contrast_limit,
    vmax=contrast_limit,
)
ax[1, 1].set_title("y-z stripe contrast")
ax[1, 1].set_xlabel("y (nm)")
ax[1, 1].set_ylabel("propagation z (nm)")
fig.colorbar(im11, ax=ax[1, 1], label=f"{refractive_index_label} minus z-slice mean")



## Multislice Propagation With Intermediate Sections

The following cell builds the selected incident field, applies the same scalar multislice operations used by the propagator, and stores the central x-z section after every material slice.

In [ ]:
beam = light_beam.beam_parameters(
    photon_energy=energy_ev,
    pol="CR",
    photon_flux=1.0,
    coherence_length=(1e-6, 1e-6),
)
wavelength = beam.wavelength

illumination = light_beam.illumination(
    beam,
    sample_shape=shape_yx,
    real_space_pixel_size=real_space_pixel_size,
)
ny, nx = shape_yx
# Convert the requested centre in metres to pixel coordinates using both axes.
# The notebook coordinate arrays use pixel centres, so the FOV centre is
# ((Ny - 1) / 2, (Nx - 1) / 2), not (Ny / 2, Ny / 2).
illumination_center_pixels = (
    np.asarray(illumination_center, dtype=float) / real_space_pixel_size
    + np.array([(ny - 1) / 2, (nx - 1) / 2], dtype=float)
)
if illumination_function in ("plane_wave", None):
    illumination.plane_wave(shape_yx)
elif illumination_function == "gaussian":
    illumination.illumination = light_beam.gauss_beam(
        sz=shape_yx,
        px_size=real_space_pixel_size,
        center=illumination_center_pixels,
        distance=illumination_focus_distance,
        fwhm=illumination_fwhm,
        wavelength=wavelength,
        alpha_beam=illumination_alpha_beam,
    )
else:
    raise ValueError(f"Unknown illumination_function: {illumination_function!r}")
scalar_field = illumination.return_illumination().astype(complex, copy=True)
if scalar_field.shape != tuple(shape_yx):
    raise ValueError(f"illumination shape {scalar_field.shape} does not match sample shape {shape_yx}")
beam_peak_yx = np.unravel_index(np.abs(scalar_field).argmax(), scalar_field.shape)
beam_peak_nm = (y_nm[beam_peak_yx[0]], x_nm[beam_peak_yx[1]])
print(
    "illumination centre pixels (y, x) = "
    f"({illumination_center_pixels[0]:.2f}, {illumination_center_pixels[1]:.2f})"
)
print(
    "nearest beam maximum (y, x) = "
    f"{beam_peak_yx}, at ({beam_peak_nm[0]:.2f}, {beam_peak_nm[1]:.2f}) nm"
)
field = light_beam.scalar_to_jones(scalar_field, polarization)
wf = object.__new__(JonesWavefronts)
wf.aperture_support_regions = None

field_after_slice = []
for iz, dz in enumerate(thicknesses):
    if hasattr(eps_stack, "project_layer"):
        k_map = JonesWavefronts.local_wavevector_directions(
            field,
            wavelength=wavelength,
            pixel_size=real_space_pixel_size,
        )
        eps_slice = eps_stack.project_layer(iz, k_map)
    else:
        eps_slice = eps_stack[iz]
    field = wf.propagate_jones_single_slice(
        field,
        eps_slice,
        wavelength,
        dz,
        aperture_support_regions=None,
    )
    field_after_slice.append(field[y0].copy())
    if iz < len(thicknesses) - 1:
        field = wf.propagate_free_space_jones(
            field,
            wavelength=wavelength,
            dz=dz,
            pixel_size=real_space_pixel_size,
        )

field_xz = np.asarray(field_after_slice)
field_xz_intensity = np.linalg.norm(field_xz, axis=-1)
field_xz_phase = np.angle(field_xz[..., 0])
detector_components = np.fft.fftshift(
    np.fft.fft2(np.fft.fftshift(field, axes=(0, 1)), axes=(0, 1)),
    axes=(0, 1),
)
exit_intensity = np.sum(np.abs(detector_components) ** 2, axis=-1)


In [ ]:
sample_boundary_xz = material_fraction[:, y0, :]
domain_boundary_xz = magnetic_domain_overlay[:, y0, :]
wedge_boundary_xz = wedge_cap_overlay[:, y0, :]

def add_optional_magnetic_borders(ax, domain_color="white", wedge_color="lime"):
    if not show_magnetic_borders_on_exit_slices:
        return
    try:
        ax.contour(
            x_nm,
            z_nm,
            domain_boundary_xz,
            levels=[0.0],
            colors=domain_color,
            linestyles="--",
            linewidths=0.8,
            alpha=0.9,
        )
    except ValueError:
        pass
    add_vector_arrows(
        ax,
        stripe_xz_x[valid_stripe_xz],
        stripe_xz_z[valid_stripe_xz],
        stripe_xz_u[valid_stripe_xz],
        stripe_xz_v[valid_stripe_xz],
        color=domain_color,
        arrow_length_nm=0.22 * magnetic_stripe_period * 1e9,
    )
    if use_wedge_caps and wedge_x > 0 and wedge_z > 0:
        try:
            ax.contour(
                x_nm,
                z_nm,
                wedge_boundary_xz,
                levels=[0.5],
                colors=wedge_color,
                linestyles="--",
                linewidths=0.9,
                alpha=0.95,
            )
        except ValueError:
            pass
        add_vector_arrows(
            ax,
            wedge_xz_x[valid_wedge_xz],
            wedge_xz_z[valid_wedge_xz],
            wedge_xz_u[valid_wedge_xz],
            wedge_xz_v[valid_wedge_xz],
            color=wedge_color,
            arrow_length_nm=0.22 * magnetic_stripe_period * 1e9,
        )

fig, ax = plt.subplots(1, 4, figsize=(18, 4), constrained_layout=True)
im_beam = ax[0].imshow(
    np.abs(scalar_field),
    origin="lower",
    extent=[x_nm[0], x_nm[-1], y_nm[0], y_nm[-1]],
)
ax[0].axvline(0.0, color="white", linestyle="--", linewidth=1.0)
ax[0].axhline(0.0, color="white", linestyle="--", linewidth=1.0)
ax[0].plot(illumination_center[1] * 1e9, illumination_center[0] * 1e9, "wo", ms=3)
ax[0].set_title("incident |field|")
ax[0].set_xlabel("x (nm)")
ax[0].set_ylabel("y (nm)")
fig.colorbar(im_beam, ax=ax[0])

im0 = ax[1].imshow(
    field_xz_intensity,
    origin="lower",
    aspect="auto",
    extent=[x_nm[0], x_nm[-1], z_nm[0], z_nm[-1]],
)
ax[1].contour(
    x_nm,
    z_nm,
    sample_boundary_xz,
    levels=[0.5],
    colors="white",
    linestyles="--",
    linewidths=1.0,
)
add_optional_magnetic_borders(ax[1], domain_color="white", wedge_color="lime")
ax[1].set_title("Jones |field| after each slice")
ax[1].set_xlabel("x (nm)")
ax[1].set_ylabel("propagation z (nm)")
fig.colorbar(im0, ax=ax[1])

im1 = ax[2].imshow(
    field_xz_phase,
    origin="lower",
    aspect="auto",
    extent=[x_nm[0], x_nm[-1], z_nm[0], z_nm[-1]],
    cmap="twilight",
    vmin=-np.pi,
    vmax=np.pi,
)
ax[2].contour(
    x_nm,
    z_nm,
    sample_boundary_xz,
    levels=[0.5],
    colors="black",
    linestyles="--",
    linewidths=1.0,
)
add_optional_magnetic_borders(ax[2], domain_color="black", wedge_color="lime")
ax[2].set_title("Jones Ex phase after each slice")
ax[2].set_xlabel("x (nm)")
ax[2].set_ylabel("propagation z (nm)")
fig.colorbar(im1, ax=ax[2])

im2 = ax[3].imshow(np.log10(exit_intensity + 1e-18), cmap="magma")
ax[3].set_title("log far-field intensity")
ax[3].set_xticks([])
ax[3].set_yticks([])
fig.colorbar(im2, ax=ax[3])
plt.show()


In [ ]:
def reconstruct(im):
    return np.fft.fftshift(np.fft.fft2(np.fft.fftshift(im)))

exit_amplitude = np.abs(field[..., 0])
exit_phase = np.angle(field[..., 0])
farfield_amplitude = np.abs(reconstruct(field[..., 0]))
farfield_log = np.log10(farfield_amplitude + 1e-18)

amp_vmin, amp_vmax = np.nanpercentile(exit_amplitude, (1, 99.5))
ff_vmin, ff_vmax = np.nanpercentile(farfield_log, (5, 99.8))

fig, ax = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)

im0 = ax[0].imshow(
    exit_amplitude,
    origin="lower",
    extent=[x_nm[0], x_nm[-1], y_nm[0], y_nm[-1]],
    cmap="magma",
    vmin=amp_vmin,
    vmax=amp_vmax,
)
ax[0].set_title("Exit wave amplitude |E_x|")
ax[0].set_xlabel("x (nm)")
ax[0].set_ylabel("y (nm)")
fig.colorbar(im0, ax=ax[0], label="amplitude")

im1 = ax[1].imshow(
    farfield_log,
    origin="lower",
    cmap="inferno",
    vmin=ff_vmin,
    vmax=ff_vmax,
)
ax[1].set_title("Far-field amplitude")
ax[1].set_xlabel("detector q_x pixel")
ax[1].set_ylabel("detector q_y pixel")
fig.colorbar(im1, ax=ax[1], label=r"$\log_{10}(|\mathcal{F}\{E_x\}|)$")

central_spectrum = farfield_amplitude[farfield_amplitude.shape[0] // 2]
central_spectrum = central_spectrum / max(np.max(central_spectrum), 1e-18)
q_px = np.arange(len(central_spectrum)) - len(central_spectrum) // 2
ax[2].plot(q_px, central_spectrum, color="tab:purple", lw=1.8)
ax[2].set_yscale("log")
ax[2].set_ylim(1e-8, 1.3)
ax[2].grid(True, which="both", alpha=0.25)
ax[2].set_title("Central far-field linecut")
ax[2].set_xlabel("q_x pixel relative to centre")
ax[2].set_ylabel("normalized amplitude")
plt.show()


Try `theta_deg = 0.0` and rerun. The material section becomes untilted, the vacuum padding disappears, and the propagation stack collapses back to the original film thickness discretization.

In [ ]:
%matplotlib inline
plt.close("all")

profiles = np.asarray(field_xz_intensity, dtype=float)
if profiles.ndim != 2:
    raise ValueError(f"field_xz_intensity must be 2D (z, x), got {profiles.shape}")

# Keep the plot readable while still sampling the full penetration depth.
line_step = max(1, int(np.ceil(profiles.shape[0] / 8)))
slice_indices = np.arange(0, profiles.shape[0], line_step)

if slice_indices[-1] != profiles.shape[0] - 1:
    slice_indices = np.append(slice_indices, profiles.shape[0] - 1)

line_cmap = plt.get_cmap("turbo")
depth_norm = plt.Normalize(vmin=float(z_nm[slice_indices].min()), vmax=float(z_nm[slice_indices].max()))
colors = line_cmap(depth_norm(z_nm[slice_indices]))

profile_scale = max(np.nanpercentile(profiles[slice_indices], 99), 1e-18)
profile_offset = 0.001

# Coherent complex line field for Fourier analysis.  Project the Jones field
# onto the incident polarization so phase variations along x contribute to the
# Fourier amplitude; do not FFT the intensity/amplitude-only line.
incident_jones = light_beam.polarization_vector(polarization)
coherent_lines = np.einsum("zxs,s->zx", field_xz, np.conjugate(incident_jones))

freq_um = np.fft.fftshift(
    np.fft.fftfreq(profiles.shape[1], d=real_space_pixel_size)
) / 1e6
spectra = []
for idx in slice_indices:
    coherent_line = coherent_lines[idx]
    spectrum = np.abs(np.fft.fftshift(np.fft.fft(np.fft.fftshift(coherent_line))))
    spectrum = spectrum / max(np.nanmax(spectrum), 1e-18)
    spectra.append(spectrum)
spectra = np.asarray(spectra)
spectrum_floor = 1e-7
spectrum_offset = 1.35

phase_lines = np.unwrap(np.angle(coherent_lines), axis=1)
phase_lines = phase_lines - np.nanmedian(phase_lines, axis=1, keepdims=True)
phase_span = max(float(np.nanpercentile(np.abs(phase_lines[slice_indices]), 95)), np.pi)
phase_offset = 2.25 * phase_span

fig, ax = plt.subplots(3, 1, figsize=(11, 11), constrained_layout=True, sharex=False)

# Gray-white magnetic-domain background for the spatial linescan panels.
# Gray marks positive film-frame OOP magnetization; white marks negative OOP.
oop_domain_xz = np.asarray(magnetization_sampled_film[:, y0, :, 2], dtype=float)
domain_material_xz = np.asarray(material_fraction[:, y0, :], dtype=float) > 0.05
if oop_domain_xz.shape != profiles.shape:
    raise ValueError(
        f"magnetic OOP x-z map must match field_xz_intensity shape; "
        f"got {oop_domain_xz.shape} and {profiles.shape}"
    )

domain_weights = domain_material_xz[slice_indices].astype(float)
domain_weight_sum = np.sum(domain_weights, axis=0)
domain_vote = np.sum(np.sign(oop_domain_xz[slice_indices]) * domain_weights, axis=0)
domain_up = domain_vote >= 0
domain_valid = domain_weight_sum > 0

# Use full-height vertical bands so the OOP domains remain visible behind all
# vertically offset lines. Gray marks positive OOP; negative OOP is left white.
def add_oop_domain_background(ax_item, alpha=0.10):
    dx_nm = float(np.mean(np.diff(x_nm))) if len(x_nm) > 1 else 1.0
    for x_center, up, valid in zip(x_nm, domain_up, domain_valid):
        if not valid:
            continue
        ax_item.axvspan(
            x_center - 0.5 * dx_nm,
            x_center + 0.5 * dx_nm,
            ymin=0.0,
            ymax=1.0,
            color="0.70" if up else "white",
            alpha=alpha,
            linewidth=0,
            zorder=-10,
        )

add_oop_domain_background(ax[0], alpha=0.10)
add_oop_domain_background(ax[1], alpha=0.10)

for line_number, (idx, color) in enumerate(zip(slice_indices, colors)):
    offset = line_number * profile_offset
    profile = profiles[idx] / profile_scale
    profile=profile/np.amax(profile)
    ax[0].plot(
        x_nm,
        profile + line_number/10,
        color=color,
        lw=1.5,
        solid_capstyle="round",
    )

ax[0].set_title("Beam amplitude linescans through the sample")
ax[0].set_xlabel("x (nm)")
ax[0].set_ylabel("normalized |E| + depth offset")
ax[0].grid(True, alpha=0.22)
ax[0].set_xlim(x_nm[0], x_nm[-1])
ax[0].set_yticks([])
ax[0].text(
    0.01,
    0.97,
    f"{len(slice_indices)} slices shown, offset = {profile_offset:.2f}",
    transform=ax[0].transAxes,
    va="top",
    ha="left",
    fontsize=9,
    bbox={"boxstyle": "round,pad=0.25", "facecolor": "white", "alpha": 0.75, "edgecolor": "none"},
)

for line_number, (idx, color) in enumerate(zip(slice_indices, colors)):
    phase = phase_lines[idx]
    ax[1].plot(
        x_nm,
        phase + 0*line_number * phase_offset,
        color=color,
        lw=1.35,
        solid_capstyle="round",
    )

ax[1].set_title("Beam phase linescans through the sample")
ax[1].set_xlabel("x (nm)")
ax[1].set_ylabel("unwrapped phase (rad) + depth offset")
ax[1].grid(True, alpha=0.22)
ax[1].set_xlim(x_nm[0], x_nm[-1])
ax[1].set_yticks([])
ax[1].text(
    0.01,
    0.97,
    f"median-centered per slice, offset = {phase_offset:.2g} rad",
    transform=ax[1].transAxes,
    va="top",
    ha="left",
    fontsize=9,
    bbox={"boxstyle": "round,pad=0.25", "facecolor": "white", "alpha": 0.75, "edgecolor": "none"},
)

for line_number, (idx, color, spectrum) in enumerate(zip(slice_indices, colors, spectra)):
    offset = line_number * spectrum_offset
    ax[2].plot(
        freq_um,
        np.log10(spectrum + spectrum_floor) + offset,
        color=color,
        lw=1.35,
        solid_capstyle="round",
    )

ax[2].set_title("Coherent Fourier amplitude of each x-linescan")
ax[2].set_xlabel(r"spatial frequency $q_x$ (cycles / $\mu$m)")
ax[2].set_ylabel(r"$\log_{10}|\mathcal{F}\{E_{\mathrm{proj}}(x)\}|$ + depth offset")
ax[2].grid(True, alpha=0.22)
ax[2].set_yticks([])

sm = plt.cm.ScalarMappable(norm=depth_norm, cmap=line_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, pad=0.015, shrink=0.92)
cbar.set_label("propagation depth z (nm)")
plt.show()
